# Qwen 3.5 397B-A17B (OpenRouter) - Full Corpus Analysis

Runs the same Variant B prompt used for Gemini against all 280 videos through Qwen 3.5-397B-A17B via OpenRouter's OpenAI-compatible API. Streams results to a JSONL log in Drive, then emits a CSV that pastes straight into the master analysis spreadsheet in the exact same layout as the Gemini output.

## What this does
- Uses the same discovered `.mp4` files from your existing corpus
- Base64-encodes each video and sends it inline via OpenRouter's `video_url` content type
- Retries JSON parse failures up to 2 times per video
- Checkpoints after every video (crash-safe — rerun and it picks up where it left off)
- Streams cost estimates live as OpenRouter returns actual usage in each response
- Emits CSV with exactly the same column layout as the Gemini output


## 1. Install and import

In [24]:
!pip install -q requests
import os, re, json, time, base64, pathlib, csv
from datetime import datetime
from collections import Counter
import requests
from google.colab import userdata, drive


## 2. Auth and Drive mount

In [25]:
API_KEY = userdata.get('OPENROUTER_API_KEY')
assert API_KEY, "Add OPENROUTER_API_KEY to Colab Secrets (left sidebar → key icon)"
drive.mount('/content/drive')
print("Auth OK. Drive mounted.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Auth OK. Drive mounted.


## 3. Config

Same corpus root as the Gemini run. `MODEL` is the OpenRouter identifier for Qwen 3.5-397B-A17B. Setting `PROVIDER_ORDER` to prefer Alibaba Cloud (cheapest and fastest per your screenshot) with fallback to other providers if it's down.


In [26]:
CORPUS_ROOT = "/content/drive/MyDrive/msc-deepfake/generated_videos_final"

GENERATOR_FOLDERS = {
    "LTX":         "ltx",
    "Hunyuan":     "hunyuan",
    "Wan":         "wan",
    "Kling":       "kling",
    "Gemini_Omni": "gemini_omni_flash",
    "Seedance":    "seedance",
    "Pexels":      "pexels",
}

EXPECTED_COUNTS = {
    "LTX": 40, "Hunyuan": 40, "Wan": 40,
    "Kling": 32, "Gemini_Omni": 32, "Seedance": 32,
    "Pexels": 64,
}

MODEL       = "qwen/qwen3.5-397b-a17b"
TEMPERATURE = 0.0
MAX_RETRIES = 2         # per-video parse retries
SLEEP_S     = 1.5       # rate-limit buffer between calls
TIMEOUT_S   = 180       # per-request timeout (video processing can take 30–90s)

# Preferred provider order (Alibaba Cloud is the cheapest & fastest per OpenRouter)
# Set to None to let OpenRouter choose (Balanced mode, price + speed)
PROVIDER_ORDER = None   # or e.g. ["Alibaba", "Chutes"] to pin

OUT_DIR  = pathlib.Path("/content/drive/MyDrive/msc-deepfake/qwen_full_run")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "qwen_results.jsonl"
CSV_PATH = OUT_DIR / "qwen_results.csv"
print(f"Log: {LOG_PATH}\nCSV: {CSV_PATH}")


Log: /content/drive/MyDrive/msc-deepfake/qwen_full_run/qwen_results.jsonl
CSV: /content/drive/MyDrive/msc-deepfake/qwen_full_run/qwen_results.csv


## 4. Discover videos and sanity-check counts

In [27]:
root = pathlib.Path(CORPUS_ROOT)
assert root.exists(), f"CORPUS_ROOT not found: {root}"

VIDEO_INDEX = []
for gen_label, subdir in GENERATOR_FOLDERS.items():
    folder = root / subdir
    if not folder.exists():
        print(f"  WARNING: {folder} does not exist — skipping {gen_label}")
        continue
    files = sorted(folder.glob("*.mp4"))
    expected = EXPECTED_COUNTS.get(gen_label, "?")
    marker = "✓" if len(files) == expected else "!"
    print(f"  {marker} {gen_label:14s} found {len(files):3d} / expected {expected}")
    for f in files:
        VIDEO_INDEX.append((gen_label, str(f)))

print(f"\nTotal videos to process: {len(VIDEO_INDEX)}")

# Also check average file size to estimate payload
sizes = [pathlib.Path(p).stat().st_size for _, p in VIDEO_INDEX]
if sizes:
    avg_kb = sum(sizes) / len(sizes) / 1024
    max_kb = max(sizes) / 1024
    # base64 adds ~33% overhead
    print(f"File sizes: avg {avg_kb:.0f} KB, max {max_kb:.0f} KB")
    print(f"After base64 encoding: avg ~{avg_kb*1.33:.0f} KB payload per request")


  ✓ LTX            found  40 / expected 40
  ✓ Hunyuan        found  40 / expected 40
  ✓ Wan            found  40 / expected 40
  ✓ Kling          found  32 / expected 32
  ✓ Gemini_Omni    found  32 / expected 32
  ✓ Seedance       found  32 / expected 32
  ✓ Pexels         found  64 / expected 64

Total videos to process: 280
File sizes: avg 452 KB, max 3029 KB
After base64 encoding: avg ~602 KB payload per request


## 5. The prompt (Variant B — same as Gemini)

In [17]:
SYSTEM_PROMPT = "You are a forensic examiner analysing a short video clip for artefacts characteristic of AI-generated content (text-to-video models). Identify which artefact families, if any, are present, using the taxonomy below. The video may or may not be AI-generated; base your judgment on evidence in the video itself.\n\n# ARTEFACT TAXONOMY (12 families)\n\n## Domain 1: Surface Artefacts (per-frame visual failures)\n1.1 Texture defects \u2014 waxy skin, plastic-looking materials, repetitive backgrounds, loss of fine detail (fabric weave, hair strands, wood grain).\n1.2 Boundary defects \u2014 blurred or soft object edges, halo/bleed around subjects, chromatic fringing, subject-background dissolution.\n1.3 Lighting inconsistency \u2014 missing or wrong-direction shadows, impossible light sources, non-physical reflections, lighting mismatched to scene.\n1.4 Watermark / provenance signal \u2014 visible watermarks, logos, or overlay text indicating generator origin (e.g. \"Sora\", \"Veo\", \"Kling\"), OR conspicuous absence-of-noise patterns and frequency-domain regularities suggestive of synthesis. Judge only from what is visible in the pixels; do not infer from filename or context.\n\n## Domain 2: Structural Defects (object and scene structure)\n2.1 Human anatomy \u2014 face defects (eyes, teeth, proportions), hand defects (finger count, grip), body issues (extra/missing limbs, impossible joints).\n2.2 Non-human anatomy \u2014 wrong limb count on animals, distorted animal faces, impossible fur/feather/scale rendering. If no animals or non-human creatures appear, return detected: false with evidence \"no non-human subjects present\".\n2.3 Object structural \u2014 distorted mechanical objects, text rendering failures (gibberish signs), wrong scale relationships, impossible topology.\n2.4 Scene composition \u2014 impossible spatial arrangements, missing expected objects, wrong perspective.\n\n## Domain 3: Temporal-Semantic Violations (across-frame, motion-visible only)\n3.1 Motion artefacts \u2014 jittery stationary objects, non-rigid motion of rigid objects, foot sliding during walking, impossible acceleration.\n3.2 Identity and object drift \u2014 face morphing between frames, clothing pattern changing, colour shifting on same object, object count changing.\n3.3 Continuity errors \u2014 objects appearing/disappearing without cause, sudden lighting shifts within a shot, background motion inconsistent with foreground, loop/repeat motion.\n3.4 Causality and physics violations \u2014 irreversibility violation (spilled liquid returning), conservation of matter violation, gravity/momentum failures, cause-effect mismatch.\n\n# SEVERITY BANDS \u2014 use the full range\n\n- L (Low): artefact present but subtle. A viewer would need to pause or look closely to notice. Example: slight waxiness on cheek skin only visible on a still frame; minor edge softness on hair.\n- M (Medium): artefact clearly present at normal playback speed and noticeable to an attentive viewer, but not the dominant feature of the frame. Example: one finger visibly merged with an adjacent finger; background text partially unreadable; a shadow direction that seems off but not impossible.\n- H (High): artefact is obvious, dominant, and would be immediately visible to a casual viewer. Example: hand with six fingers or fingers melting into each other; a person's face morphing shape mid-shot; an object phasing through a solid surface; text that is complete gibberish across the whole sign.\n\n# CONFIDENCE CALIBRATION (integer 0\u2013100)\n\nUse the full range. Anchors:\n- 0\u201320: I am confident this video is Real. No credible AI markers.\n- 30\u201350: Genuinely uncertain. Some markers either way but nothing decisive.\n- 60\u201380: Probably AI-generated. Multiple clear markers but some ambiguity.\n- 85\u2013100: Almost certainly AI-generated. Multiple severe artefacts or a watermark.\n\nA batch of responses all clustered at 60\u201375 indicates you are hedging. Use the extremes when the evidence warrants.\n\nDo NOT default to M. If the artefact is subtle, use L. If it is dominant and unmistakable, use H. A response where every detected family is \"M\" is almost certainly miscalibrated \u2014 reconsider.\n\n# OUTPUT FORMAT\n\nRespond ONLY with the JSON below. No preamble, no code fences, no trailing commentary.\n\n{\n  \"video_verdict\": \"AI-generated\" | \"Real\" | \"Uncertain\",\n  \"confidence\": <integer 0-100>,\n  \"families\": {\n    \"1.1_texture\":          {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"<observation with timestamp 0:XX and spatial location>\"},\n    \"1.2_boundary\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.3_lighting\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.4_watermark\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.1_human_anatomy\":    {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.2_non_human_anatomy\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.3_object_structural\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.4_scene_composition\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.1_motion\":           {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.2_identity_drift\":   {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.3_continuity\":       {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.4_causality\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"}\n  },\n  \"most_diagnostic_artefact\": \"<one sentence naming the single most decisive observation, or 'no strong artefacts observed'>\",\n  \"notes\": \"<one to two sentences of additional observation, or empty string>\"\n}\n\n# RULES\n\n1. Mark \"detected\": true only when you observe specific evidence in the video, not on suspicion.\n2. When \"detected\" is false, set \"severity\": \"none\" and \"evidence\" to a brief reason.\n3. Every \"evidence\" field for a detected artefact must include (a) a timestamp 0:XX and (b) a spatial location (e.g. \"bottom-left\", \"on the subject's right hand\", \"in the mirror reflection\", \"across the whole frame\").\n4. \"Real\" is a valid verdict. Do not assume AI-generation.\n5. Use the full L/M/H range as calibrated above.\n6. Output must be valid JSON, all 12 family keys present, no additional keys, no markdown."
print(f"Prompt: {len(SYSTEM_PROMPT)} chars, ~{len(SYSTEM_PROMPT)//4} tokens")


Prompt: 6378 chars, ~1594 tokens


## 6. Analyser with retry and parse validation

In [29]:
FAMILIES = ["1.1_texture", "1.2_boundary", "1.3_lighting", "1.4_watermark",
            "2.1_human_anatomy", "2.2_non_human_anatomy", "2.3_object_structural",
            "2.4_scene_composition", "3.1_motion", "3.2_identity_drift",
            "3.3_continuity", "3.4_causality"]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

def extract_json(text):
    if not text or not isinstance(text, str):
        return None
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    t = t.strip()
    obj = None
    try:
        obj = json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.DOTALL)
        if m:
            try: obj = json.loads(m.group(0))
            except Exception: return None
        else:
            return None
    if isinstance(obj, list):
        obj = obj[0] if obj else None
    if not isinstance(obj, dict) or "families" not in obj:
        return None
    return obj

def has_full_schema(obj):
    fams = obj.get("families")
    return isinstance(fams, dict) and all(f in fams for f in FAMILIES)

def video_to_data_uri(video_path):
    with open(video_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:video/mp4;base64,{b64}"

def call_openrouter(video_path, system_prompt):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "video_url",
                     "video_url": {"url": video_to_data_uri(video_path)}},
                    {"type": "text",
                     "text": "Analyse this video. Respond with ONLY the JSON object specified in the system instruction. All 12 family keys must be present. No markdown fences, no code blocks, no preamble."},
                ],
            },
            # Prime the response to force the exact schema
            {"role": "assistant", "content": '{"video_verdict":'},
        ],
        "temperature": TEMPERATURE,
        "response_format": {"type": "json_object"},
        "provider": {
            "order": ["Alibaba", "Chutes"],
            "allow_fallbacks": False,   # do NOT fall back to Novita
        },
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://kent.ac.uk",
        "X-Title": "MSc Dissertation - Video Artefact Analysis",
    }

    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=TIMEOUT_S)
    resp.raise_for_status()
    data = resp.json()

    if "error" in data:
        raise RuntimeError(f"OpenRouter error: {data['error']}")
    if "choices" not in data or not data["choices"]:
        raise RuntimeError(f"No choices in response: {data}")

    raw_content = data["choices"][0]["message"]["content"]
    # Re-attach the primer since we forced the assistant to start with '{"video_verdict":'
    raw_text = '{"video_verdict":' + raw_content if not raw_content.strip().startswith('{') else raw_content

    u = data.get("usage", {}) or {}
    usage = {
        "input_tokens":  u.get("prompt_tokens"),
        "output_tokens": u.get("completion_tokens"),
        "total_tokens":  u.get("total_tokens"),
        "cost_usd":      u.get("cost"),
        "provider":      data.get("provider"),
    }
    return raw_text, usage

def analyse_video_with_retries(video_path, system_prompt):
    last_raw, last_usage = "", {}
    for attempt in range(MAX_RETRIES + 1):
        try:
            raw, usage = call_openrouter(video_path, system_prompt)
            last_raw, last_usage = raw, usage
            obj = extract_json(raw)
            if obj is not None and has_full_schema(obj):
                return {"status": "ok", "attempt": attempt + 1, "parsed": obj,
                        "raw_response": raw, "usage": usage}
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else "?"
            body = e.response.text[:400] if e.response is not None else ""
            if code == 429:
                wait = 5 * (attempt + 1)
                print(f"  rate-limited (429), backing off {wait}s")
                time.sleep(wait)
                continue
            if code in (500, 502, 503, 504):
                wait = 3 * (attempt + 1)
                print(f"  server error {code}, retry after {wait}s")
                time.sleep(wait)
                continue
            # Non-retryable HTTP error
            return {"status": "http_error", "attempt": attempt + 1, "parsed": None,
                    "raw_response": body, "usage": {}, "error": f"HTTP {code}: {body[:200]}"}
        except Exception as e:
            last_error = str(e)
    return {"status": "parse_failed", "attempt": MAX_RETRIES + 1,
            "parsed": None, "raw_response": last_raw, "usage": last_usage}


In [22]:
# Smoke test- one call, full prompt, show output shape
raw, usage = call_openrouter(VIDEO_INDEX[0][1], SYSTEM_PROMPT)
print(f"Provider: {usage.get('provider')}")
print(f"Cost: ${usage.get('cost_usd')}")
print(f"Tokens: in={usage.get('input_tokens')} out={usage.get('output_tokens')}")
print(f"\n=== First 800 chars of response ===")
print(raw[:800])

parsed = extract_json(raw)
if parsed:
    print(f"\nParsed OK. Families present: {list(parsed.get('families', {}).keys())}")
    print(f"Schema complete: {has_full_schema(parsed)}")
else:
    print("\nPARSE FAILED — response wasn't valid JSON")

Provider: Alibaba
Cost: $0.00309231
Tokens: in=2727 out=867

=== First 800 chars of response ===
{
  "video_verdict": "AI-generated",
  "confidence": 95,
  "families": {
    "1.1_texture": {
      "detected": true,
      "severity": "M",
      "evidence": "At 0:00, the skin on the subject's face appears overly smooth and waxy, lacking natural pore detail, particularly on the forehead and cheeks."
    },
    "1.2_boundary": {
      "detected": true,
      "severity": "M",
      "evidence": "At 0:01, the edges of the subject's hair blend indistinctly into the background foliage, creating a soft, melting halo effect rather than sharp separation."
    },
    "1.3_lighting": {
      "detected": false,
      "severity": "none",
      "evidence": "Lighting appears consistent with an outdoor sunny environment; no obvious directional conflicts observed."
    },
    "1.4_watermark": {
      "de

Parsed OK. Families present: ['1.1_texture', '1.2_boundary', '1.3_lighting', '1.4_watermark', '2.1_hu

## 7. Main run - checkpointed

Rerun this cell to resume if Colab disconnects. It reads `qwen_results.jsonl`, skips anything already logged, and picks up from there.


In [30]:
# Load checkpoint
done_ids = set()
if LOG_PATH.exists():
    with LOG_PATH.open() as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                rec = json.loads(line)
                done_ids.add(rec.get("video_id"))
            except Exception:
                pass
print(f"Checkpoint: {len(done_ids)} videos already logged. Will skip them.")

# Live tallies
n_ok = n_fail = 0
total_cost = 0.0
total_in = total_out = 0
t_start = time.time()

remaining = [(g, p) for g, p in VIDEO_INDEX if pathlib.Path(p).stem not in done_ids]
print(f"To process this run: {len(remaining)}\n")

with LOG_PATH.open("a") as f:
    for i, (gen_label, video_path) in enumerate(remaining, start=1):
        vid = pathlib.Path(video_path).stem
        print(f"[{i:3d}/{len(remaining)}] {gen_label:12s} {vid[:50]:50s}", end="  ")
        t0 = time.time()
        try:
            result = analyse_video_with_retries(video_path, SYSTEM_PROMPT)
            record = {
                "video_id": vid,
                "generator": gen_label,
                "prompt_variant": "B_calibrated",
                "model": MODEL,
                "mllm": "Qwen_3.5",
                "status": result["status"],
                "attempts": result["attempt"],
                "elapsed_s": round(time.time() - t0, 1),
                "usage": result["usage"],
                "parsed": result["parsed"],
                "raw_response": result["raw_response"],
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            }
            if "error" in result:
                record["error"] = result["error"]

            if result["status"] == "ok":
                n_ok += 1
            else:
                n_fail += 1

            u = result["usage"] or {}
            if u.get("cost_usd") is not None:
                total_cost += u["cost_usd"]
            if u.get("input_tokens"):
                total_in += u["input_tokens"]
            if u.get("output_tokens"):
                total_out += u["output_tokens"]

            elapsed_min = (time.time() - t_start) / 60
            print(f"{result['status']:12s}  a={result['attempt']}  "
                  f"({record['elapsed_s']}s)  ok={n_ok} fail={n_fail}  "
                  f"${total_cost:.2f}  {elapsed_min:.1f}m")
        except Exception as e:
            record = {
                "video_id": vid,
                "generator": gen_label,
                "prompt_variant": "B_calibrated",
                "model": MODEL,
                "mllm": "Qwen_3.5",
                "status": "error",
                "error": str(e)[:400],
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            }
            n_fail += 1
            print(f"ERROR: {str(e)[:80]}")

        f.write(json.dumps(record) + "\n")
        f.flush()
        os.fsync(f.fileno())   # force write to Drive to survive Colab kernel death
        time.sleep(SLEEP_S)

print(f"\n=== Run complete ===")
print(f"OK: {n_ok}  Failed: {n_fail}")
print(f"Tokens — input: {total_in:,}  output: {total_out:,}")
print(f"Total cost this run: ${total_cost:.2f}")


Checkpoint: 0 videos already logged. Will skip them.
To process this run: 280

[  1/280] LTX          w2_001_ltx_20260719_105436                          ok            a=1  (7.4s)  ok=1 fail=0  $0.00  0.1m
[  2/280] LTX          w2_002_ltx_20260719_105611                          ok            a=1  (7.4s)  ok=2 fail=0  $0.01  0.3m
[  3/280] LTX          w2_003_ltx_20260719_105639                          ok            a=1  (6.6s)  ok=3 fail=0  $0.01  0.4m
[  4/280] LTX          w2_004_ltx_20260719_105708                          ok            a=1  (6.3s)  ok=4 fail=0  $0.01  0.5m
[  5/280] LTX          w2_005_ltx_20260719_105736                          ok            a=1  (7.7s)  ok=5 fail=0  $0.01  0.7m
[  6/280] LTX          w2_006_ltx_20260719_105804                          ok            a=1  (6.5s)  ok=6 fail=0  $0.01  0.8m
[  7/280] LTX          w2_007_ltx_20260719_105832                          ok            a=1  (9.8s)  ok=7 fail=0  $0.02  1.0m
[  8/280] LTX          w2_008_lt

## 8. Convert JSONL → CSV

Column names match `MLLM_Analysis_Log.xlsx` exactly and match your Gemini CSV, so you can concatenate or paste alongside.


In [31]:
BASE_COLS = ["video_id", "generator", "mllm", "batch_date", "verdict", "confidence"]
FAM_COLS = []
for key in FAMILIES:
    FAM_COLS += [f"{key}__detected", f"{key}__severity", f"{key}__evidence"]
TAIL_COLS = ["most_diagnostic_artefact", "notes", "review_flag"]
ALL_COLS = BASE_COLS + FAM_COLS + TAIL_COLS

records = []
with LOG_PATH.open() as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: records.append(json.loads(line))
        except Exception: pass

rows = []
for rec in records:
    parsed = rec.get("parsed") or {}
    fams = parsed.get("families", {}) if isinstance(parsed, dict) else {}
    row = {
        "video_id": rec.get("video_id"),
        "generator": rec.get("generator"),
        "mllm": "Qwen_3.5",
        "batch_date": (rec.get("timestamp") or "")[:10],
        "verdict": parsed.get("video_verdict", ""),
        "confidence": parsed.get("confidence", ""),
        "most_diagnostic_artefact": parsed.get("most_diagnostic_artefact", ""),
        "notes": parsed.get("notes", ""),
        "review_flag": "" if rec.get("status") == "ok" else f"STATUS: {rec.get('status')}",
    }
    for key in FAMILIES:
        fam = fams.get(key) if isinstance(fams, dict) else None
        if isinstance(fam, dict):
            det = fam.get("detected")
            row[f"{key}__detected"] = "TRUE" if det is True else ("FALSE" if det is False else "")
            row[f"{key}__severity"] = fam.get("severity", "")
            row[f"{key}__evidence"] = fam.get("evidence", "")
        else:
            row[f"{key}__detected"] = ""
            row[f"{key}__severity"] = ""
            row[f"{key}__evidence"] = ""
    rows.append(row)

with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=ALL_COLS)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

print(f"Wrote {len(rows)} rows to {CSV_PATH}")


Wrote 280 rows to /content/drive/MyDrive/msc-deepfake/qwen_full_run/qwen_results.csv


## 9. Summary - what got done, what needs attention

In [32]:
by_gen_status = {}
for rec in records:
    key = (rec.get("generator", "?"), rec.get("status", "?"))
    by_gen_status[key] = by_gen_status.get(key, 0) + 1

gens = sorted({g for g, _ in by_gen_status})
statuses = sorted({s for _, s in by_gen_status})
print(f"{'Generator':14s}  " + "  ".join(f"{s:>14s}" for s in statuses) + "   total   expected")
for g in gens:
    counts = {s: by_gen_status.get((g, s), 0) for s in statuses}
    total = sum(counts.values())
    exp = EXPECTED_COUNTS.get(g, "?")
    print(f"{g:14s}  " + "  ".join(f"{counts[s]:>14d}" for s in statuses) + f"   {total:>5d}   {exp}")

# Verdict distribution among successful runs
ok_recs = [r for r in records if r.get("status") == "ok"]
verdicts = Counter((r.get("parsed") or {}).get("video_verdict") for r in ok_recs)
print(f"\nVerdicts (n={len(ok_recs)}): {dict(verdicts)}")

failed = [r for r in records if r.get("status") != "ok"]
if failed:
    print(f"\n{len(failed)} records need attention:")
    for r in failed[:10]:
        print(f"  {r.get('generator'):12s} {r.get('video_id','')[:55]:55s} {r.get('status')}  {str(r.get('error',''))[:80]}")
    if len(failed) > 10:
        print(f"  ... and {len(failed)-10} more")
    print("\nTo retry failures: delete their lines from qwen_results.jsonl and rerun Cell 7.")


Generator                   ok   total   expected
Gemini_Omni                 32      32   32
Hunyuan                     40      40   40
Kling                       32      32   32
LTX                         40      40   40
Pexels                      64      64   64
Seedance                    32      32   32
Wan                         40      40   40

Verdicts (n=280): {'AI-generated': 84, 'Real': 194, 'Uncertain': 2}
